<a href="https://colab.research.google.com/github/Lagnadeep-samal/Machine-learning-models/blob/main/End_to_End.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score,roc_curve,confusion_matrix,accuracy_score,recall_score,precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer

In [2]:
diabetic_data=pd.read_csv('/content/diabetes.csv')
diabetic_data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
diabetic_data.shape

(768, 9)

In [4]:
diabetic_data.isna().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


In [5]:
diabetic_data.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

In [17]:
diabetic_data['Outcome'].value_counts()

,count
Outcome,
0,500
1,268


In [38]:
Q1 = diabetic_data["Glucose"].quantile(0.25)

Q3 = diabetic_data["Glucose"].quantile(0.75)

IQR = Q3 - Q1


lower_bound = Q1 - 1.5 * IQR

upper_bound = Q3 + 1.5 * IQR


outliers = diabetic_data[
    (diabetic_data["Glucose"]< lower_bound) |
    (diabetic_data["Glucose"] > upper_bound)
]


print("Number of outliers:", len(outliers))

Number of outliers: 0


In [7]:
cols_with_invalid_zeros=[
    'Glucose',
    'BloodPressure',
    "SkinThickness",
    'Insulin',
    'BMI'
]
for col in cols_with_invalid_zeros:
  diabetic_data[col]=diabetic_data[col].replace(0,np.nan)

In [8]:
X=diabetic_data.drop('Outcome',axis=1)
y=diabetic_data['Outcome']

In [9]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [10]:
diabetic_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   763 non-null    float64
 2   BloodPressure             733 non-null    float64
 3   SkinThickness             541 non-null    float64
 4   Insulin                   394 non-null    float64
 5   BMI                       757 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(6), int64(3)
memory usage: 54.1 KB


In [41]:
from sklearn.impute import SimpleImputer
numeric_features=X.columns.to_list()
numeric_pipeline=Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy='median')),
        ("scaler",RobustScaler())
    ]
)

In [42]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num', numeric_pipeline, numeric_features)
    ]
)

In [43]:
model_pipeline = Pipeline(
    steps = [
        ('preprocessing', preprocessor),
        ('model', LogisticRegression())
    ]
)

In [44]:
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['Pregnancies', 'Glucose',
                                                   'BloodPressure',
                                                   'SkinThickness', 'Insulin',
                                                   'BMI',
                                                   'DiabetesPedigreeFunction',
                                                   'Age'])])),
                ('model', LogisticRegression())])

In [45]:
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:, 1]

In [46]:
y_pred

array([1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0,
       1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1,
       0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0])

In [47]:
accuracy_score(y_test,y_pred)

0.7012987012987013

In [28]:
recall_score(y_test,y_pred)

0.5

In [29]:
precision_score(y_test,y_pred)

0.6

In [30]:
confusion_matrix(y_test,y_pred)

array([[82, 18],
       [27, 27]])

In [31]:
roc_auc_score(y_test,y_prob)

np.float64(0.812962962962963)

In [32]:
import joblib
joblib.dump(model_pipeline,"diabetes_pipeline.pkl")


['diabetes_pipeline.pkl']

In [34]:
loaded_model = joblib.load('/content/diabetes_pipeline.pkl')
y_pred_loaded_model = loaded_model.predict(X_test)
y_pred_loaded_model

array([1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0,
       1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1,
       0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0])

In [33]:
custom_threshold=0.3
y_custom_pred=(y_prob>=custom_threshold).astype(int)
recall_score(y_test,y_custom_pred)

0.7962962962962963

control the data imbalance


In [36]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler()

X_train_over, y_train_over = ros.fit_resample(
    X_train,
    y_train
)

print(y_train_over.value_counts())

Outcome
0    400
1    400
Name: count, dtype: int64
